<a href="https://colab.research.google.com/github/LeonimerMelo/Reinforcement-Learning/blob/Policy-Gradient/Introdu%C3%A7%C3%A3o_ao_TD3_(Twin_Delayed_Deep_Deterministic_Policy_Gradient)_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução ao TD3 (Twin Delayed Deep Deterministic Policy Gradient)

O **TD3 (Twin Delayed Deep Deterministic Policy Gradient)** é um algoritmo moderno de **Reinforcement Learning (RL)** baseado em **Actor-Critic**, criado para melhorar o desempenho do **DDPG (Deep Deterministic Policy Gradient)** em ambientes com **ações contínuas**.

O **Twin Delayed DDPG (TD3)** é um algoritmo de *reinforcement learning* (aprendizado por reforço) projetado especificamente para lidar com **espaços de ação contínuos**, ou seja, situações onde as ações que o agente pode tomar não são discretas (como "esquerda" ou "direita"), mas sim valores contínuos (como "aplicar uma força de 0.75 Newtons em um ângulo de 30 graus"). Ele é um sucessor direto do DDPG (Deep Deterministic Policy Gradient) e foi proposto para corrigir suas principais fragilidades, resultando em um treinamento mais estável e confiável.

Ele é muito utilizado em problemas como:

* Controle de robôs
* Simulação física
* Controle de veículos
* Jogos com ações contínuas
* Ambientes como `Pendulum`, `HalfCheetah`, `Walker2d`, `BipedalWalker`

O TD3 pertence à família dos algoritmos **Off-Policy**, ou seja:

* O agente aprende usando experiências armazenadas em memória.
* A política atual pode ser diferente da política usada para coletar dados.

---

## 🧠 O Problema que o TD3 Resolve: A Superestimação

O DDPG, seu predecessor, sofre de um problema comum em algoritmos que usam Q-learning: a **superestimação do valor das ações**. Como o agente usa uma estimativa para atualizar a si mesmo, ele tende a acreditar que suas ações são melhores do que realmente são, acumulando erros que podem levar a políticas subótimas ou até mesmo ao colapso do aprendizado.

O TD3 introduz três inovações principais para combater esse problema, que são explicitamente refletidas em seu nome e estrutura:

### 1. O "Twin" (Gêmeo): Clipped Double Q-Learning

Em vez de usar uma única rede neural para estimar o valor (Q-value) de um par estado-ação, o TD3 usa **duas redes "críticas" independentes**. Ambas recebem o mesmo estado e ação, mas calculam o valor Q separadamente.

Durante a atualização, o TD3 usa o **menor** dos dois valores para calcular o alvo do aprendizado. Essa abordagem, conhecida como *clipped double Q-learning*, é uma maneira eficaz de evitar a superestimação ao longo do treinamento.

### 2. O "Delayed" (Atrasado): Delayed Policy Updates

No TD3, a rede do **ator** (a política que decide a ação) é atualizada com **menos frequência** do que as redes críticas. Enquanto os críticos são atualizados a cada passo de treinamento, o ator é atualizado a cada dois passos (sendo `policy_delay = 2` um hiperparâmetro comum).

Isso estabiliza o treinamento porque garante que a política seja atualizada apenas quando as estimativas de valor (dos críticos) já estiverem mais estáveis, reduzindo a variância do gradiente da política.

### 3. Target Policy Smoothing

Para adicionar robustez, o TD3 adiciona um pequeno **ruído** à ação da política alvo (`target_policy_noise`) e recorta esse ruído para um valor máximo (`target_noise_clip`). Isso suaviza a função Q alvo, tornando o aprendizado mais resistente a pequenas imprecisões e evitando que o agente explore picos estreitos na função de valor.

---

## 1. Revisão rápida: Reinforcement Learning

Em Reinforcement Learning temos:

* **Agente**: quem toma decisões
* **Ambiente**: mundo onde o agente atua
* **Estado** $(s_t)$: situação atual
* **Ação** $(a_t)$: decisão tomada
* **Recompensa** $(r_t)$: retorno recebido
* **Próximo estado** $(s_{t+1})$

O objetivo é maximizar a recompensa acumulada:

$$
R_t =
r_t+\gamma r_{t+1}+\gamma^2 r_{t+2}+...
$$

onde: $\gamma$ é o fator de desconto.

---

# 2. O problema das ações contínuas

Em problemas discretos podemos fazer:

Exemplo:

```
Ações:
0 = esquerda
1 = direita
2 = parado
```

Mas em robótica:

```
Torque do motor = 0.374
Ângulo = -0.82
Velocidade = 1.52
```

As ações são números reais.

Nesse caso usamos algoritmos como:

* DDPG
* TD3
* SAC

---

# 3. O DDPG (base do TD3)

O DDPG usa duas redes:

## Actor

Responsável por escolher ações.

$$
a = \pi(s)
$$

Exemplo:

Entrada:

```
posição = 0.5
velocidade = -0.2
```

Saída:

```
torque = 0.73
```

---

## Critic

Avalia a ação.

Ele estima:

$$
Q(s,a)
$$

que representa:

"Qual o valor de tomar esta ação neste estado?"

---

Estrutura:

```
Estado
   |
   v
 Actor
   |
   v
 Ação
   |
   +--------+
            |
            v
          Critic
            |
            v
        Valor Q
```

---

# 4. Problema do DDPG: superestimação

O DDPG possui um problema:

O Critic pode aprender valores muito altos.

Exemplo:

Valor verdadeiro:

```
Q = 5.0
```

Mas o Critic aprende:

```
Q = 8.7
```

Isso causa políticas ruins.

Esse problema é chamado:

**Overestimation Bias**

---

# 5. Ideia principal do TD3

O TD3 introduz três melhorias principais:

---

# Melhoria 1 — Dois Critics (Twin Critics)

Ao invés de um Critic:

```
Critic 1
```

usa:

```
Critic 1
Critic 2
```

Eles calculam:

$$
Q_1(s,a)
$$

e

$$
Q_2(s,a)
$$

O TD3 usa o menor valor:

$$
Q_{target}=min(Q_1,Q_2)
$$

Exemplo:

Critic 1:

```
Q = 8.5
```

Critic 2:

```
Q = 5.7
```

O TD3 escolhe:

```
Q = 5.7
```

Isso reduz superestimação.

---

# Melhoria 2 — Target Policy Smoothing

O TD3 adiciona ruído na ação futura:

$$
a'=\pi(s')+\epsilon
$$

onde:

$$
\epsilon \sim N(0,\sigma)
$$

Exemplo:

Ação prevista:

```
0.80
```

Com ruído:

```
0.74
```

Isso evita que o Critic memorize ações muito específicas.

---

# Melhoria 3 — Delayed Policy Update

O Actor não é atualizado sempre.

No DDPG:

```
Critic update
Actor update
Critic update
Actor update
```

No TD3:

```
Critic update
Critic update
Critic update
Actor update
```

Normalmente:

$$
d=2
$$

ou seja:

a cada 2 atualizações do Critic, atualiza o Actor.

---

# 6. Arquitetura TD3 completa

![Image](https://images.openai.com/static-rsc-4/ahGQmrXsesPUkVYWFpvmAz69a-jsSxRrpGc7gPk_CaXBhsm-jrtQjg-5wLp2m7A6Ug_yzjT5WeDLGjUW6sMFNBWIIc6ARcBvMKrhq4kVgP3SihlBjws0RDoJ9oFRbVRqs-_mTnNq6kP52QEFv62O-HBePGtRY5AZGiGbDtJMxzCVBpJ-4ewqlXCI7nAbkzzz?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/_XyalJfyZIZtrd-VXo85RkWAcC-P288rf514FzAk3O3BN55qEBxYWWJaEhtEdRChVZcfSuJ5CPyN8mv_-utKTnkFo7Hrh1Xrb2lq5nncr-kcZaUI23QVSNTwK48-s5brJe-SxYVV9DNEUHar8Tgaqu7v39MjfdYSlft_OEDPYUeCqzsNpUZO8fAmmNSvM4l1?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/Tjd83Rurynq33WaqYq7Utu5mj7PnWMM2yTzE0Sh83QIgcYuUub9qT_nG0SghgQZ2zTtlaoF8iFhJACzG7meuSoI-5YhdjaN17rPPM_V66FlvzMMFceX-TKIYGQwAVXRJLxePsNxFy5FfwZHtCoqE6uiipasgN3mCvomxiefUJO4qSEjmJSwbgiH7JZApYmYC?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/7haHSGE2rWiGEFGNn9GpdmD3FgpyYgc8SdS7TpCs6xxAg9WFRDsUXQ1nIFeFNFC2qIjXafIcbZ0eMbh56hPAqjjqueeHSatCkrZo3lFWIjMi_HCqQsyFKSr_Kj1q5iIMysqnkju9jedy1fQdPr_MIYHigilQbBvXH9fJO2_9G2psWx0VAcZOz5PdlCYOQ41z?purpose=fullsize)

Fluxo:

```
              Estado s
                 |
                 |
              Actor
                 |
                 v
             Ação a
                 |
                 |
      +----------+----------+
      |                     |
      v                     v
  Critic 1              Critic 2
      |                     |
      +----------+----------+
                 |
              min(Q1,Q2)

```

---

# 7. Equações do TD3

## Atualização do Critic

O alvo:

$$
y =
r+\gamma
min(Q_1'(s',a'),Q_2'(s',a'))
$$

O erro:

$$
Loss=
(Q(s,a)-y)^2
$$

O Critic minimiza esse erro.

---

## Atualização do Actor

O Actor tenta maximizar:

$$
J(\theta)=
E[Q(s,\pi(s))]
$$

Na prática:

```python
loss_actor = -critic(state, actor(state))
```

---

# 8. Implementação didática em PyTorch

## Redes neurais

```python
import torch
import torch.nn as nn
import torch.optim as optim


class Actor(nn.Module):

    def __init__(self,state_dim,action_dim):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,action_dim),
            nn.Tanh()
        )


    def forward(self,x):
        return self.net(x)
```

---

## Critic duplo

```python
class Critic(nn.Module):

    def __init__(self,state_dim,action_dim):

        super().__init__()

        self.q1 = nn.Sequential(
            nn.Linear(state_dim+action_dim,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,1)
        )


        self.q2 = nn.Sequential(
            nn.Linear(state_dim+action_dim,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,1)
        )


    def forward(self,state,action):

        x=torch.cat([state,action],1)

        return self.q1(x),self.q2(x)
```

---

# 9. Replay Buffer

O TD3 aprende de experiências antigas:

$$
(s,a,r,s',done)
$$

Exemplo:

```python
from collections import deque
import random


buffer = deque(maxlen=100000)


def store(s,a,r,s2,d):

    buffer.append(
        (s,a,r,s2,d)
    )


def sample(batch):

    data=random.sample(buffer,batch)

    return zip(*data)
```

---

# 10. Treinamento do Critic

```python
with torch.no_grad():

    next_action = target_actor(next_state)

    noise=torch.randn_like(next_action)*0.2

    next_action += noise


    q1,q2 = target_critic(
        next_state,
        next_action
    )


    target = reward + gamma * torch.min(q1,q2)
```

Agora treinamos:

```python
q1,q2 = critic(state,action)

loss = (
    (q1-target)**2 +
    (q2-target)**2
).mean()


critic_optimizer.zero_grad()

loss.backward()

critic_optimizer.step()
```

---

# 11. Atualização do Actor atrasada

A cada 2 passos:

```python
if step % 2 == 0:


    action = actor(state)


    actor_loss = -critic.q1(
        state,
        action
    ).mean()


    actor_optimizer.zero_grad()

    actor_loss.backward()

    actor_optimizer.step()
```

---

# 12. Ambiente exemplo Gymnasium

Instalando:

```bash
pip install gymnasium[classic-control]
```

Exemplo:

```python
import gymnasium as gym


env=gym.make(
    "Pendulum-v1"
)


state,_=env.reset()


print(state)

```

Saída:

```
[
cos(theta),
sin(theta),
velocity
]
```

Ação:

```
[-2 , 2]
```

Torque contínuo.

---

# 13. Comparação DDPG vs TD3

| Característica       | DDPG   | TD3      |
| -------------------- | ------ | -------- |
| Actor                | Sim    | Sim      |
| Critic               | 1      | 2        |
| Reduz overestimation | Não    | Sim      |
| Ruído alvo           | Não    | Sim      |
| Atualização Actor    | sempre | atrasada |
| Estabilidade         | média  | alta     |

---

# 14. Intuição simples

Imagine dois avaliadores:

DDPG:

```
Avaliador único:

"Essa ação vale 10!"
```

O agente acredita.

TD3:

```
Avaliador 1:
vale 10

Avaliador 2:
vale 6

TD3:
vou considerar 6
```

O agente fica mais conservador e aprende melhor.

---

# 15. Quando usar TD3?

Excelente para:

✅ Controle contínuo

✅ Robótica

✅ Simulações físicas

✅ Alta dimensionalidade

Exemplos:

* Robôs manipuladores
* Braços mecânicos
* Carros autônomos simulados
* MuJoCo
* PyBullet

Menos indicado para:

❌ Ações discretas simples

(Nesses casos usar DQN, Double DQN, PPO etc.)




<center><img src='https://drive.google.com/uc?id=1XWgIAl_w10flmXMYZoRfb_2TXrtom-_e' width=1200></center>

## 🧬 Código Explicativo

A implementação do TD3 segue a estrutura clássica de algoritmos *actor-critic* off-policy. Aqui estão os componentes centrais explicados:

### 1. Inicialização do Agente

Neste estágio, todos os componentes principais são criados:

*   **Actor:** A política que mapeia estados para ações.
*   **Critic_1 e Critic_2:** As duas redes Q.
*   **Target Actor:** Uma cópia do actor para estabilidade.
*   **Target Critic_1 e Target Critic_2:** Cópias das redes críticas.

```python
class TD3:
    def __init__(self, state_dim, action_dim, max_action):
        # Componentes principais do agente
        self.actor = Actor(state_dim, action_dim, max_action)
        self.actor_target = Actor(state_dim, action_dim, max_action)
        self.actor_target.load_state_dict(self.actor.state_dict())

        self.critic_1 = Critic(state_dim, action_dim)
        self.critic_2 = Critic(state_dim, action_dim)
        self.critic_1_target = Critic(state_dim, action_dim)
        self.critic_2_target = Critic(state_dim, action_dim)
        
        # Carregamento dos pesos para as targets
        self.critic_1_target.load_state_dict(self.critic_1.state_dict())
        self.critic_2_target.load_state_dict(self.critic_2.state_dict())
```
*A estrutura de um agente TD3 em PyTorch inclui um ator e dois críticos, além de suas respectivas redes alvo*.

### 2. Seleção da Ação

Durante a exploração, o TD3 adiciona ruído à ação escolhida pela política para incentivar a descoberta de diferentes estratégias. Durante a avaliação, o agente usa a política de forma determinística, sem ruído.

```python
def select_action(self, state):
    # Seleciona a ação do ator (determinística)
    action = self.actor(state)
    
    # Adiciona ruído para exploração durante o treinamento
    if self.exploration_noise is not None:
        noise = np.random.normal(0, self.exploration_noise, size=action_dim)
        action = (action + noise).clip(self.min_action, self.max_action)
    
    return action
```
*A seleção de ação envolve a adição de ruído gaussiano à ação determinística do ator para encorajar a exploração*.

### 3. Atualização do Agente (O Coração do TD3)

O processo de treinamento é onde as três inovações do TD3 se manifestam.

1.  **Amostragem:** Uma mini-batch de transições é amostrada do *replay buffer*.
2.  **Target Policy Smoothing:** O ruído é adicionado à ação do *target actor*.
3.  **Cálculo do Alvo:** O valor Q alvo é calculado como o **mínimo** entre as saídas dos dois *target critics*.
4.  **Atualização dos Críticos:** Ambos os críticos são atualizados para minimizar o erro entre suas previsões e o valor alvo calculado.
5.  **Atualização Atrasada do Ator:** Se o número de passos for múltiplo de `policy_delay`, o ator e as redes alvo são atualizados.

```python
def train(self, replay_buffer, batch_size, discount_factor=0.99, tau=0.005, policy_delay=2):
    # Amostra uma mini-batch de transições do buffer
    state, action, reward, next_state, done = replay_buffer.sample(batch_size)

    with torch.no_grad():
        # 1. Target Policy Smoothing: adiciona ruído à ação do target actor
        noise = (torch.randn_like(action) * self.target_noise).clamp(-self.noise_clip, self.noise_clip)
        next_action = (self.actor_target(next_state) + noise).clamp(-self.max_action, self.max_action)

        # 2. Twin Critics: escolhe o mínimo valor Q dos dois alvos
        target_Q1 = self.critic_1_target(next_state, next_action)
        target_Q2 = self.critic_2_target(next_state, next_action)
        target_Q = torch.min(target_Q1, target_Q2)
        
        # Calcula o valor alvo final
        target_Q = reward + (1 - done) * discount_factor * target_Q

    # 3. Atualiza ambos os críticos para minimizar o erro
    current_Q1 = self.critic_1(state, action)
    current_Q2 = self.critic_2(state, action)
    critic_loss = F.mse_loss(current_Q1, target_Q) + F.mse_loss(current_Q2, target_Q)
    self.critic_optimizer.zero_grad()
    critic_loss.backward()
    self.critic_optimizer.step()

    # 4. Delayed Policy Update: Atualiza o ator a cada 'policy_delay' passos
    if self.total_it % policy_delay == 0:
        # Calcula a perda do ator, que é o negativo do valor Q médio previsto pelo crítico 1
        actor_loss = -self.critic_1(state, self.actor(state)).mean()
        
        # Atualiza o ator
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # 5. Soft update das redes alvo (Polyak averaging)
        for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
            target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)
        # ... (mesmo procedimento para os críticos)
```
*O loop de treinamento do TD3 aplica as três principais correções: clipping duplo dos Q-values, suavização do ruído na política alvo e atualização atrasada da política*.

---

## ⚖️ TD3 vs. Outros Algoritmos RL

Para entender o TD3, é crucial compará-lo com outros algoritmos importantes:

| Característica | **TD3** | **DDPG** (Predecessor) | **SAC (Soft Actor-Critic)** | **PPO (Proximal Policy Optimization)** |
| :--- | :--- | :--- | :--- | :--- |
| **Tipo** | Off-policy, Actor-Critic | Off-policy, Actor-Critic | Off-policy, Actor-Critic | On-policy, Actor-Critic |
| **Espaço de Ação** | Contínuo | Contínuo | Contínuo | Contínuo ou Discreto |
| **Principal Força** | **Estável e Robusto**, combate a superestimação | Simples e eficaz em muitos problemas | **Alta eficiência amostral** e exploração guiada por entropia | **Estável e confiável**, com atualizações de política seguras |
| **Principal Fraqueza** | Pode ser menos eficiente que o SAC em alguns casos | **Sofre com superestimação** e alta variância | **Complexo** e com muitos hiperparâmetros a ajustar | **Baixa eficiência amostral** (on-policy) e pode ser mais lento |
| **Exploração** | Ruído na ação (Gaussiano) | Ruído na ação (Gaussiano/OU) | Entropia máxima (exploração guiada) | Estocástica (distribuição de probabilidade sobre ações) |
| **Nº de Críticos** | **2** (Twin Critics) | 1 | 2 (geralmente) | 1 (função de valor) |
| **Atraso na Política** | **Sim** (Delayed Updates) | Não | Não | Não |
| **Exemplo de Uso** | Navegação de robôs, controle de processos | Controle contínuo em geral | Tarefas complexas com alta dimensionalidade | Tarefas onde estabilidade é crítica, como jogos e finanças |

### Em Resumo

*   **TD3 vs. DDPG:** O TD3 é um DDPG aprimorado que corrige sua principal falha (superestimação) com as três técnicas mencionadas (críticos gêmeos, atraso e suavização).
*   **TD3 vs. SAC:** Ambos são algoritmos off-policy de ponta para ação contínua. O SAC é geralmente mais eficiente em termos de amostras e explora melhor o ambiente, mas é mais complexo. O TD3 é frequentemente mais estável e mais fácil de ajustar em alguns ambientes de controle.
*   **TD3 vs. PPO:** PPO é um algoritmo on-policy, o que o torna mais estável, mas também menos eficiente com os dados. TD3, sendo off-policy, pode reutilizar experiências passadas, tornando-o mais eficiente. A escolha entre eles depende do equilíbrio desejado entre estabilidade e eficiência amostral.

O TD3 é um dos algoritmos mais robustos e amplamente utilizados para problemas de controle contínuo, representando um avanço significativo na estabilidade e confiabilidade do aprendizado por reforço.

#Referências
A principal referência para o algoritmo **TD3 (Twin Delayed Deep Deterministic Policy Gradient)** é o artigo seminal de seus criadores.

Esse artigo introduziu o TD3 e as três técnicas fundamentais: **Twin Critics, Delayed Policy Updates e Target Policy Smoothing**.

### 📚 Referência Fundamental (O Artigo Original)

**FUJIMOTO, Scott; HOOF, Herke van; MEGER, David.** Addressing Function Approximation Error in Actor-Critic Methods. In: **International Conference on Machine Learning**, 2018. p. 1582-1591.

*   **Nota:** Este é o artigo onde o TD3 foi proposto, introduzindo as três inovações centrais (Twin Critics, Delayed Policy Updates e Target Policy Smoothing) para lidar com a superestimação do valor da ação em algoritmos Actor-Critic. As citações em formato BibTeX e as referências a este trabalho são comuns em implementações do algoritmo .

### 🔬 Aplicações Acadêmicas (Exemplos de Uso)

O TD3 tem sido aplicado com sucesso em diversas áreas de pesquisa. Os exemplos abaixo demonstram sua utilização em problemas complexos de otimização, servindo como referências para aplicações práticas.

**Energia e Sistemas de Potência:**

**CHEN, Siwei; LI, Jianjun; ZOU, Xinxun; et al.** Low-carbon Economic Dispatch of Electric-thermal Coupling System Based on Twin Delayed DDPG Reinforcement Learning. **Modern Electric Power**, v. 42, n. 2, p. 314-321, 2025. DOI: 10.19725/j.cnki.1007-2322.2023.0058.

**Comunicações sem Fio e Redes:**

**IEEE.** Energy Efficient RIS-Assisted UAV Networks Using Twin Delayed DDPG Technique. **IEEE Transactions on Wireless Communications**, v. 23, n. 12, p. 18423-18439, dez. 2024.

**Processamento de Sinais e Fala:**

**KHAN, Muhammad Salman; GUL, Sania.** STEM: spatial speech separation using twin-delayed DDPG reinforcement learning and expectation maximization. **Applied Acoustics**, Elsevier, 2025. DOI: 10.1016/j.apacoust.2025.111022.